[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/svngoku/cybersup-deep-learning-intro/blob/main/notebooks/formateur/04_attention_masques_corrige.ipynb)


# TP 04 — Attention calculée et testée
**150 minutes · NumPy puis PyTorch · CPU**

Objectifs : retrouver les poids, suivre les axes multi-têtes et démontrer la causalité par un test.
Découpage : calcul 25 min, axes 45 min, masques 35 min, gradients 25 min, restitution 20 min.

**VERSION FORMATEUR — réponses et points de contrôle en fin de notebook.**

## Préparation de l'environnement

Exécuter cette cellule en premier dans un runtime Python 3 sur **CPU**. Les bibliothèques déjà installées sont conservées ; seules celles qui manquent sont ajoutées. Aucun clonage, fichier voisin ou montage Drive n'est nécessaire. Enregistrer une copie du notebook avant de commencer. Si le lien vers le dépôt privé est inaccessible, importer le fichier fourni dans le pack étudiant.


In [ ]:
# Cellule autonome : les bibliothèques déjà présentes dans Colab sont conservées.
import importlib.util
import json
import os
import platform
import subprocess
import sys

DEPENDENCIES = {
    "numpy": "numpy==2.2.6",
    "sklearn": "scikit-learn==1.7.2",
    "torch": "torch==2.8.0",
    "matplotlib": "matplotlib==3.10.6",
}
missing = [pin for module, pin in DEPENDENCIES.items()
           if importlib.util.find_spec(module) is None]
if missing:
    print("Installation des bibliothèques absentes :", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])

import matplotlib
import numpy as np
import sklearn
import torch

ENVIRONMENT = {
    "runtime": "google-colab" if "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ else "python-local-ou-ci",
    "python": platform.python_version(),
    "numpy": np.__version__,
    "scikit_learn": sklearn.__version__,
    "torch": torch.__version__,
    "matplotlib": matplotlib.__version__,
    "device": "cpu",
    "cuda_available": torch.cuda.is_available(),
}
print(json.dumps(ENVIRONMENT, indent=2, ensure_ascii=False))
print("Les cinq TP utilisent le CPU, même si un GPU est disponible.")


In [ ]:
import os, copy, math, random
import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits, make_moons
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(2)
device = torch.device('cpu')
print('Référence : CPU | PyTorch', torch.__version__, '| graine', SEED)

In [ ]:
Q=np.eye(2); K=np.eye(2); V=np.diag([1.,2.])
scores=Q@K.T/np.sqrt(2)
def softmax(a):
    e=np.exp(a-a.max(axis=-1,keepdims=True)); return e/e.sum(axis=-1,keepdims=True)
A=softmax(scores); O=A@V
print('Scores :',scores,'Poids :',A,'Sorties :',O,sep='\n')
assert np.allclose(A.sum(-1),1)
assert np.allclose(A,[[.66976155,.33023845],[.33023845,.66976155]])
causal=np.triu(np.ones((2,2),dtype=bool),1)
Am=softmax(np.where(causal,-np.inf,scores))
assert np.allclose(Am[0],[1,0])
print('Attention causale :',Am,'Sortie causale :',Am@V,sep='\n')

In [ ]:
class ManualMHA(nn.Module):
    def __init__(self,d=8,heads=2):
        super().__init__(); assert d%heads==0
        self.d=d;self.heads=heads;self.dh=d//heads
        self.q=nn.Linear(d,d);self.k=nn.Linear(d,d);self.v=nn.Linear(d,d);self.out=nn.Linear(d,d)
    def forward(self,x,forbidden=None):
        B,T,D=x.shape
        def split(proj): return proj(x).reshape(B,T,self.heads,self.dh).transpose(1,2)
        q,k,v=split(self.q),split(self.k),split(self.v)
        s=q@k.transpose(-2,-1)/math.sqrt(self.dh)
        if forbidden is not None: s=s.masked_fill(forbidden,float('-inf'))
        a=torch.softmax(s,dim=-1)
        context=(a@v).transpose(1,2).reshape(B,T,D)
        return self.out(context),a

model=ManualMHA(); x=torch.randn(2,4,8,requires_grad=True)
mask=torch.triu(torch.ones(4,4,dtype=torch.bool),diagonal=1)
out,weights=model(x,mask)
assert out.shape==(2,4,8) and weights.shape==(2,2,4,4)
assert torch.allclose(weights.sum(-1),torch.ones(2,2,4))
assert weights[:,:,0,1:].abs().max().item()==0
out.square().mean().backward()
assert x.grad is not None and torch.isfinite(x.grad).all()
assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in model.parameters())

In [ ]:
model.eval(); changed=x.detach().clone(); changed[:,2:]+=100
with torch.inference_mode():
    base,_=model(x.detach(),mask); altered,_=model(changed,mask)
    unmasked,_=model(x.detach()); unmasked_changed,_=model(changed)
causal_error=(base[:,:2]-altered[:,:2]).abs().max().item()
unmasked_error=(unmasked[:,:2]-unmasked_changed[:,:2]).abs().max().item()
assert causal_error<1e-6
assert unmasked_error>1e-5 # contrôle négatif : le test détecte bien l'absence de masque
RESULT={'tp':4,'causal_prefix_error':causal_error,'unmasked_prefix_change':unmasked_error,
        'weights_shape':list(weights.shape)}
print(RESULT)

## Investigations
1. Refaire le calcul pour une valeur V différente, sans changer Q ou K. Que devient A ?
2. Ajouter un masque de padding en veillant à garder au moins une clé accessible.
3. Permuter les tokens et vérifier l'équivariance en l'absence de masque et de positions.
4. Changer le nombre de têtes en gardant d=8. Quelles formes changent ? Quels paramètres restent constants ?
5. Expliquer pourquoi des poids d'attention ne sont pas une preuve causale complète d'une décision.

## Corrigé — invariants et pièges
A≈[[0,66976155;0,33023845],[0,33023845;0,66976155]]. O≈[[0,66976155;0,6604769],[0,33023845;1,3395231]].
Changer V seul ne change pas A. Un masque causal force la première ligne à [1,0].
Les colonnes n'ont pas à sommer à un. Une ligne toute à −inf mène à un softmax non défini : NaN.
Sans position ni masque, SA(PX)=P SA(X), les poids de projection étant partagés entre positions.
Le contrôle négatif sans masque montre que le test de causalité est discriminant.
Le code utilise True=interdit ; vérifier la convention de toute API de remplacement.

## Exporter les résultats

Exécuter la cellule suivante après le bilan terminal pour enregistrer métriques, graine et versions dans un fichier JSON. Dans Colab, activer `DOWNLOAD_RESULTS` pour le télécharger. Sauvegarder également une copie du notebook avec les sorties et les réponses aux investigations. Les fichiers du runtime sont temporaires.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path

DOWNLOAD_RESULTS = False  # @param {type:"boolean"}

def json_scalar(value):
    if isinstance(value, np.generic):
        return value.item()
    raise TypeError(f"Valeur non sérialisable : {type(value).__name__}")

result_path = Path("resultats") / f"tp_{RESULT['tp']:02d}.json"
result_path.parent.mkdir(exist_ok=True)
experiment = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "seed": SEED,
    "environment": ENVIRONMENT,
    "results": RESULT,
}
result_path.write_text(
    json.dumps(experiment, indent=2, ensure_ascii=False, default=json_scalar) + "\n",
    encoding="utf-8",
)
print("Résultats enregistrés :", result_path)

if DOWNLOAD_RESULTS:
    try:
        from google.colab import files
    except ImportError:
        print("Hors Colab, récupérer le fichier dans le dossier resultats/.")
    else:
        files.download(str(result_path))
